# Plant Disease Detection - Jalgaon Regional Model Training

This notebook is designed to train a deep learning model for detecting diseases in crops specific to the Jalgaon region in Maharashtra: **Banana, Cotton, Brinjal, and Maize**.

### Instructions:
1. **Enable GPU**: Go to `Runtime` -> `Change runtime type` -> `Hardware accelerator` -> `T4 GPU`.
2. **Upload Dataset**: 
   - Create a folder structure like this: `dataset/train/Banana_Healthy`, `dataset/train/Banana_Sigatoka`, etc.
   - Zip the `dataset` folder and upload it to the Colab files sidebar.
3. **Run All**: Execute all cells to train the model and generate the `jalgaon_disease_model.pt` file.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np
import os

# 1. Unzip the dataset (Upload dataset.zip first)
import shutil
if os.path.exists('dataset.zip'):
    !unzip -q dataset.zip -d ./data

In [ ]:
# 2. Data Preparation
data_dir = './data/dataset'
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(os.path.join(data_dir, 'train'), transform=transform)
val_dataset = datasets.ImageFolder(os.path.join(data_dir, 'val'), transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

class_names = train_dataset.classes
print(f"Detected Classes: {class_names}")

In [ ]:
# 3. Model Definition (Transfer Learning with ResNet18)
# Using 'weights' as 'pretrained' is deprecated
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(class_names))

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
print(f"Training on: {device}")
if device.type == 'cpu':
    print("*** WARNING: GPU NOT DETECTED! Training will be VERY slow. Go to Runtime -> Change runtime type -> T4 GPU! ***")

In [ ]:
# 4. Training Loop
epochs = 10
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    
    print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(train_dataset):.4f}")

In [ ]:
# 5. Evaluation and Confusion Matrix
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - Jalgaon Crops')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# 6. Save Model
torch.save(model.state_dict(), 'jalgaon_disease_model.pt')
print("Model saved as jalgaon_disease_model.pt")